In [1]:
import torch 
import torch.nn as nn
from torch.optim import Adam
import lightning as L
from torch.utils.data import TensorDataset, DataLoader

## Building an LSTM

In [ ]:
class myLSTM(L.LightningModule):
    def __init__(self):
        super().__init__()
        L.seed_everything(seed=42)

        # We initialize using Normal Distribution
        mean = torch.tensor(0.0)
        std = torch.tensor(1.0)

        # All biases are initialized to 0

        # These are the Weights and Biases in the first stage
        self.wlr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wlr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.blr1 = nn.Parameter(torch.tensor(0.), requires_grad=True)

        # These are the Weights and Biases in the second stage
        self.wpr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wpr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bpr1 = nn.Parameter(torch.tensor(0.), requires_grad=True)

        self.wp1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wp2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bp1 = nn.Parameter(torch.tensor(0.), requires_grad=True)

        # These are the Weightsd and Biases in the third stage.
        self.wo1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wo2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bo1 = nn.Parameter(torch.tensor(1.), requires_grad=True)


        # If we wanted to initialize using Uniform Distribution.
        # self.wlr1 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.wlr2 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.blr1 = nn.Parameter(torch.rand(1), requires_grad=True)

        # self.wpr1 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.wpr2 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.bpr1 = nn.Parameter(torch.rand(1), requires_grad=True)

        # self.wp1 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.wp2 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.bp1 = nn.Parameter(torch.rand(1), requires_grad=True)

        # self.wo1 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.wo2 = nn.Parameter(torch.rand(1), requires_grad=True)
        # self.bo1 = nn.Parameter(torch.rand(1), requires_grad=True)

    def lstm_unit(self, input_value, long_memory, short_memory):

        # 1) In the first stage determines what percentage of the current long-term memory should be remembered
        long_remember_percent = torch.sigmoid((short_memory * self.wlr1) + (input_value * self.wlr2) + self.blr1)

        # 2) The second stage creates a new, potential long-term memory 
        # and determines what percenage of that to add to the current long-term memory
        potential_remember_percent = torch.sigmoid((short_memory * self.wpr1) + (input_value * self.wpr2) + self.bpr1)

        potential_memory = torch.tanh((short_memory * self.wp1) + (input_value * self.wp2) + self.bp1)

        # Once we have gone through the first two stages, we update the long-term memory
        update_long_memory = ((long_memory * long_remember_percent) + (potential_remember_percent * potential_memory))

        # 3) The third stage creates a new, potential short-term memory 
        # and determines what percentage of that should be remembered and used as output
        output_percent = torch.sigmoid((short_memory * self.wo1) + (input_value * self.wo2) + self.bo1)

        updated_short_memory = torch.tanh(update_long_memory) * output_percent

        # Finally, we return the updated long and short-term memories
        return ([update_long_memory, updated_short_memory])
    

    def forward(self, input):
        long_memory = 0
        short_memory = 0
        day1 = input[0]
        day2 = input[1]
        day3 = input[2]
        day4 = input[3]

        # Day 1
        long_memory, short_memory = self.lstm_unit(day1, long_memory, short_memory)

        # Day2
        long_memory, short_memory = self.lstm_unit(day2, long_memory, short_memory)

        # Day3
        long_memory, short_memory = self.lstm_unit(day3, long_memory, short_memory)

        # Day4
        long_memory, short_memory = self.lstm_unit(day4, long_memory, short_memory)

        # Return short memory which is the output of the LSTM
        return short_memory 
    

    def configure_optimizers(self):
        return Adam(self.parameters())
    

    def training_step(self, batch): # take a step during gradient descent
        input_i, label_i = batch # collect inputs
        output_i = self.forward(input_i[0])
        loss = (output_i - label_i)**2

        self.log("train_loss", loss, on_step=True, on_epoch=True)
        if (label_i == 0):
            self.log("out_0", output_i, on_step=True)
        else:
            self.log("out_1", output_i, on_step=True)
        return loss

In [3]:
model = myLSTM()

print("Before optimization, the parameters are... ")
for name, param in model.named_parameters():
    print(name, param.data)

print("Comparisson between the observed and predicted values...")
print("Company A: Observed = 0, Predicted = ", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 0, Predicted = ", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

Seed set to 42


Before optimization, the parameters are... 
wlr1 tensor(0.3367)
wlr2 tensor(0.1288)
blr1 tensor(0.)
wpr1 tensor(0.2345)
wpr2 tensor(0.2303)
bpr1 tensor(0.)
wp1 tensor(-1.1229)
wp2 tensor(-0.1863)
bp1 tensor(0.)
wo1 tensor(2.2082)
wo2 tensor(-0.6380)
bo1 tensor(1.)
Comparisson between the observed and predicted values...
Company A: Observed = 0, Predicted =  tensor(-0.0611)
Company B: Observed = 0, Predicted =  tensor(-0.0612)


## We Create a DataLoader

In [4]:
inputs = torch.tensor([[0., 0.5, 0.25, 1.], [1., 0.5, 0.25, 1.]])
lables = torch.tensor([0., 1.])

dataset = TensorDataset(inputs, lables)
dataloader = DataLoader(dataset)

In [5]:
trainer = L.Trainer(max_epochs=2000)
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name         | Type | Params | Mode | FLOPs
-----------------------------------------------------
  | other params | n/a  | 12     | n/a  | n/a  
-----------------------------------------------------
12        Trainable params
0         Non-trainable params
12        Total params
0.000     Total estimated model params size (MB)
0         Modules in train mode
0         Modules in eval mode
0         Total Flops
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python

Epoch 1999: 100%|██████████| 2/2 [00:00<00:00, 194.46it/s, v_num=0]

`Trainer.fit` stopped: `max_epochs=2000` reached.


Epoch 1999: 100%|██████████| 2/2 [00:00<00:00, 102.50it/s, v_num=0]


In [6]:
print("Comparisson between the observed and predicted values...")
print("Company A: Observed = 0, Predicted = ", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 0, Predicted = ", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())


Comparisson between the observed and predicted values...
Company A: Observed = 0, Predicted =  tensor(0.4651)
Company B: Observed = 0, Predicted =  tensor(0.5858)


In [ ]:
# We need a lot more training but before we do just to make sure training improves prediction. 
# We will plot the loss values and predictions saved in log files with TensorBoard first. 
# on terminal we run 'tensorboard --logdir=lightning_logs/' and then go to http://localhost:6006/ 


## Optimizing (Training) the Weights and Biases in the LSTM 

In [8]:
# Adding another 1000 epoches
path_to_checkpoint = trainer.checkpoint_callback.best_model_path
trainer = L.Trainer(max_epochs=3000)
trainer.fit(model, train_dataloaders=dataloader, ckpt_path=path_to_checkpoint)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Restoring states from the checkpoint path at /Users/odysseasgeorgiades/Desktop/Neural Networks/06/lightning_logs/version_0/checkpoints/epoch=1999-step=4000.ckpt
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/Users/odysseasgeorgiades/Desktop/Neural Networks/06/lightning_logs/version_0/checkpoints' to '/Users/odysseasgeorgiades/Desktop/Neural Networks/06/lightning

Epoch 2999: 100%|██████████| 2/2 [00:00<00:00, 189.04it/s, v_num=1]

`Trainer.fit` stopped: `max_epochs=3000` reached.


Epoch 2999: 100%|██████████| 2/2 [00:00<00:00, 101.65it/s, v_num=1]


In [9]:
print("Comparisson between the observed and predicted values...")
print("Company A: Observed = 0, Predicted = ", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 0, Predicted = ", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

Comparisson between the observed and predicted values...
Company A: Observed = 0, Predicted =  tensor(0.3669)
Company B: Observed = 0, Predicted =  tensor(0.6709)


In [10]:
# Adding another 2000 epoches
path_to_checkpoint = trainer.checkpoint_callback.best_model_path
trainer = L.Trainer(max_epochs=5000)
trainer.fit(model, train_dataloaders=dataloader, ckpt_path=path_to_checkpoint)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Restoring states from the checkpoint path at /Users/odysseasgeorgiades/Desktop/Neural Networks/06/lightning_logs/version_1/checkpoints/epoch=2999-step=6000.ckpt
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/Users/odysseasgeorgiades/Desktop/Neural Networks/06/lightning_logs/version_1/checkpoints' to '/Users/odysseasgeorgiades/Desktop/Neural Networks/06/lightning

Epoch 4999: 100%|██████████| 2/2 [00:00<00:00, 194.79it/s, v_num=2]

`Trainer.fit` stopped: `max_epochs=5000` reached.


Epoch 4999: 100%|██████████| 2/2 [00:00<00:00, 104.12it/s, v_num=2]


In [11]:
print("Comparisson between the observed and predicted values...")
print("Company A: Observed = 0, Predicted = ", model(torch.tensor([0., 0.5, 0.25, 1.])).detach())
print("Company B: Observed = 0, Predicted = ", model(torch.tensor([1., 0.5, 0.25, 1.])).detach())

Comparisson between the observed and predicted values...
Company A: Observed = 0, Predicted =  tensor(0.0039)
Company B: Observed = 0, Predicted =  tensor(0.9603)


In [12]:
# Now Company A is close to 0 and Company B is close to 1.

# We print the final estimated Weights and Biases. 
print("After optimization, the parameters are...")
for name, param in model.named_parameters():
    print(name, param.data)

After optimization, the parameters are...
wlr1 tensor(2.7293)
wlr2 tensor(1.5747)
blr1 tensor(1.6192)
wpr1 tensor(1.8638)
wpr2 tensor(1.5161)
bpr1 tensor(0.4932)
wp1 tensor(1.2810)
wp2 tensor(0.8609)
bp1 tensor(-0.2909)
wo1 tensor(4.1505)
wo2 tensor(-0.5871)
bo1 tensor(1.1585)
